In [ ]:
import os, re, glob, json, io, hashlib
from datetime import datetime
import pandas as pd
import psycopg2
from psycopg2 import sql

# ---------- CONFIG dinámico (usa env vars dentro del contenedor) ----------
BRONZE_ROOT = os.environ.get("BRONZE_PATH", "/app/scripts/bronze/outputs")
MAPPINGS_FILE = os.environ.get("MAPPINGS_FILE", "/app/scripts/silver/categories_mapping.json")
DB_DSN = os.environ.get("DB_DSN", "host=postgres_db port=5432 dbname=myapp user=admin password=secure_password")
# --------------------------------------------------------------------------

In [ ]:
def create_schema_and_tables(conn):
    cur = conn.cursor()
    cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS silver.categories(
            snapshot_date DATE NOT NULL,
            supermarket TEXT NOT NULL,
            
            -- Jerarquía de categorías
            category_lvl1_id TEXT,
            category_lvl1_name TEXT,
            category_lvl1_slug TEXT,
            
            category_lvl2_id TEXT,
            category_lvl2_name TEXT,
            category_lvl2_slug TEXT,
            
            category_lvl3_id TEXT,
            category_lvl3_name TEXT,
            category_lvl3_slug TEXT,
            
            -- Metadata
            created_at TIMESTAMP DEFAULT NOW()--,

            --PRIMARY KEY (snapshot_date, supermarket, category_lvl1_name, category_lvl2_name, category_lvl3_name)
        );
    """)
    conn.commit()
    cur.close()

In [ ]:
def full_load_categories(conn):
    # 1️⃣ Leer el mapping
    with open(MAPPINGS_FILE, "r") as f:
        mappings = json.load(f)
    
    cur = conn.cursor()
    cur.execute("TRUNCATE TABLE IF EXISTS silver.categories;")
    conn.commit()

    for sup, conf in mappings.items():
        path = os.path.join(BRONZE_ROOT, conf["path"])
        print(f"\nProcesando {sup} desde {path}")

        files = glob.glob(os.path.join(path, "*.csv")) + glob.glob(os.path.join(path, "*.json"))
        if not files:
            print(f"⚠️ No se encontraron archivos para {sup}")
            continue
        
        for fpath in files:
            print(f" - Archivo: {fpath}")

            # 2️⃣ Leer archivo según tipo
            if conf["type"] == "csv":
                df = pd.read_csv(fpath)
            else:
                df = pd.read_json(fpath)

            # 3️⃣ Aplicar mapping
            rename_map = conf["columns"]
            df = df.rename(columns=rename_map)

            # 4️⃣ Agregar columnas comunes
            df["supermarket"] = sup
            # Extraer snapshot_date del nombre de archivo (asumiendo formato _YYYY-MM-DD)
            snapshot = re.findall(r"\d{4}-\d{2}-\d{2}", fpath)
            df["snapshot_date"] = snapshot[0] if snapshot else datetime.now().date()
            df["created_at"] = datetime.now()

            # 5️⃣ Filtrar solo las columnas del esquema
            cols_keep = [
                "snapshot_date", "supermarket",
                "category_lvl1_id", "category_lvl1_name", "category_lvl1_slug",
                "category_lvl2_id", "category_lvl2_name", "category_lvl2_slug",
                "category_lvl3_id", "category_lvl3_name", "category_lvl3_slug",
                "created_at"
            ]
            for col in cols_keep:
                if col not in df.columns:
                    df[col] = None
            df = df[cols_keep]

            # 6️⃣ Insertar con COPY (mucho más eficiente que INSERT)
            buf = io.StringIO()
            df.to_csv(buf, header=False, index=False, sep="\t", na_rep="\\N")
            buf.seek(0)
            cur.copy_from(buf, "silver.categories", sep="\t", null="\\N", columns=cols_keep)
            conn.commit()

    cur.close()
    print("\n✅ Full load completado correctamente.")

In [ ]:
if __name__ == "__main__":
    conn = connect_to_postgres()
    if conn:
        full_load_categories(conn)
        conn.close()